# FSOC Virtual Camera Tracking System — Colab Runner

Coarse-alignment simulator for mobile Free Space Optical Communication (FSOC) terminals (SIH Problem Statement 4).

This notebook:
1. Clones/installs the `fsoc_tracker` package from this GitHub repo
2. Runs the virtual camera tracking simulation with configurable disturbances
3. Renders the tracking video inline and saves it as `.mp4`
4. Plots real-time performance metrics (tracking error, lock status, FPS)
5. Generates the automated performance report required as a deliverable

**Before running:** replace `REPO_URL` below with your own GitHub repo URL after you push this project.

In [ ]:
# @title 1. Setup: clone repo and install dependencies
REPO_URL = "https://github.com/<your-username>/fsoc-virtual-tracking.git"  # ← update this

import os
if not os.path.exists("fsoc-virtual-tracking"):
    !git clone $REPO_URL
%cd fsoc-virtual-tracking
!pip install -q -e .
!pip install -q opencv-python-headless matplotlib pyyaml

In [ ]:
# @title 1b. (Alternative) Skip GitHub — upload project as a zip instead
# If you don't want to use git clone, comment out cell 1 above and instead:
#   1. Upload fsoc-virtual-tracking.zip using the Colab file browser (left sidebar)
#   2. Run this cell to unzip and install it
#
# from google.colab import files
# uploaded = files.upload()  # choose fsoc-virtual-tracking.zip
# !unzip -oq fsoc-virtual-tracking.zip
# %cd fsoc-virtual-tracking
# !pip install -q -e .
print("Skip this cell if you used git clone in Cell 1.")

In [ ]:
# @title 2. Imports
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

from fsoc_tracker.simulator import Simulation, DEFAULT_CONFIG
print("fsoc_tracker package loaded OK.")

In [ ]:
# @title 3. Configure the scenario  (edit values, then run)
target_motion = "circular"        # @param ["straight_line","circular","figure8","random","spiral","sinusoidal"]
target_size = 10                  # @param {type:"slider", min:5, max:20, step:1}
target_speed = 150                # @param {type:"slider", min:20, max:400, step:10}
noise_type = "gaussian"           # @param ["none","gaussian","salt_pepper","poisson"]
atmosphere = "clear"              # @param ["clear","haze","fog","rain","low_light"]
platform_motion = "linear"        # @param ["linear","circular","random","spiral","figure8"]
camera_jitter_px = 5              # @param {type:"slider", min:0, max:20, step:1}
num_frames = 300                  # @param {type:"slider", min:60, max:1200, step:60}

cfg = dict(DEFAULT_CONFIG)
cfg.update({
    "target_motion": target_motion,
    "target_size": target_size,
    "target_speed": target_speed,
    "noise_types": [] if noise_type == "none" else [noise_type],
    "atmosphere": atmosphere,
    "platform_motion": platform_motion,
    "camera_jitter_px": camera_jitter_px,
})
sim = Simulation(cfg)
print("Simulation configured:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

In [ ]:
# @title 4. Run the simulation and record frames + metrics
frames = []
err_history, locked_history, time_history = [], [], []

def on_frame(result, i):
    vis = Simulation.annotate(result["frame"], result)
    frames.append(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    time_history.append(result["sim_time"])
    locked_history.append(1 if result["locked"] else 0)
    gt = result["gt_frame_pos"]
    if gt is not None and result["locked"]:
        tx, ty = result["track_pos"]
        err_history.append(float(np.hypot(tx - gt[0], ty - gt[1])))
    else:
        err_history.append(np.nan)

summary = sim.run(num_frames, on_frame=on_frame)
print("Run complete.\n")
for k, v in summary.items():
    print(f"{k:28s}: {v}")

In [ ]:
# @title 5. Play the tracking video inline
fig = plt.figure(figsize=(6, 4.5))
plt.axis("off")
im = plt.imshow(frames[0])

def update(i):
    im.set_array(frames[i])
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000/30, blit=True)
plt.close(fig)
display(HTML(ani.to_jshtml()))

In [ ]:
# @title 6. Save the tracking run as an MP4 file
out_path = "tracking_output.mp4"
h, w = frames[0].shape[:2]
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 30, (w, h))
for f in frames:
    writer.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
writer.release()
print(f"Saved: {out_path}")

# Uncomment to download directly in Colab:
# from google.colab import files
# files.download(out_path)

In [ ]:
# @title 7. Plot performance metrics (tracking error, lock status)
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(time_history, err_history, color="tab:blue")
axes[0].axhline(10, color="red", linestyle="--", label="Spec limit (10 px)")
axes[0].set_ylabel("Tracking error (px)")
axes[0].legend()
axes[0].set_title("Tracking Error over Time")

axes[1].fill_between(time_history, locked_history, step="mid", color="tab:green", alpha=0.6)
axes[1].set_ylabel("Locked (1/0)")
axes[1].set_xlabel("Simulation time (s)")
axes[1].set_title("Lock Status over Time")
plt.tight_layout()
plt.savefig("performance_plot.png", dpi=150)
plt.show()

In [ ]:
# @title 8. Generate mandatory Performance Log (CSV + report.txt)
sim.perf.save_csv("performance_log.csv")
sim.perf.save_report("performance_report.txt")
print(open("performance_report.txt").read())

In [ ]:
# @title 9. (Optional) Benchmark-2: run tracking on a provided .mp4 file instead of the virtual camera
# Upload a video file (e.g. from the SIH benchmark set) and this cell will run the
# detector + Kalman tracker directly on those frames, bypassing the virtual PTZ camera,
# exactly as required by the 'Benchmark Performance-2' evaluation stage.

from fsoc_tracker.detector import BeaconDetector
from fsoc_tracker.tracker import CentroidKalmanTracker

VIDEO_PATH = ""  # @param {type:"string"}

if VIDEO_PATH:
    cap = cv2.VideoCapture(VIDEO_PATH)
    det = BeaconDetector()
    trk = CentroidKalmanTracker()
    bench_frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        prior = trk.pos if trk.locked else None
        d = det.detect(frame, prior_pos=prior)
        pos, locked = trk.update((d["cx"], d["cy"]) if d else None)
        vis = frame.copy()
        color = (0, 255, 0) if locked else (0, 0, 255)
        cv2.circle(vis, (int(pos[0]), int(pos[1])), 12, color, 2)
        bench_frames.append(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    cap.release()
    print(f"Processed {len(bench_frames)} frames from {VIDEO_PATH}")
else:
    print("Set VIDEO_PATH to run Benchmark-2 video evaluation.")